In [29]:
import pandas as pd
from utils import ngram_utils, gpt2_utils, grnn_utils
from utils.text_utils import *

## Testing sampling of continuations

In [2]:
print(ngram_utils.sample_continuation("My name is "))
print(ngram_utils.sample_continuation("I want a "))
print(ngram_utils.sample_continuation("The cat sat on the"))

liaisons inconvenience importing factories gathering locusts colonia dossier randomly pottery kellogg sloping reincorporated hummingbird 3.06 grandson gauges goodison patio guderian
neill co-ordinate rectum kippur sevens holtzman single-sex ics lower-case rodney veritable nat refinery martha 2009 conn frets lilith oss dictators
xiao 1474 kenner waitangi 1602 nascent coco ethic dispatching maid madero diminished capri mont radio-television clearing co-op arrowheads deliberations ripple


In [4]:
print(grnn_utils.sample_continuation("My name is "))
print(grnn_utils.sample_continuation("I want a "))
print(grnn_utils.sample_continuation("The cat sat on the"))

sometimes credited as the <unk> .
break down with the total return to <unk> and <unk> ; or family " cold material " for the duration
wall as they pour through coastal cities , outer walls , foggy valley , tall sand beaches and anything else


In [9]:
print("Sentence 1")
print(gpt2_utils.sample_continuation("My name is "))
print("Sentence 2")
print(gpt2_utils.sample_continuation("I want a "))
print("Sentence 3")
print(gpt2_utils.sample_continuation("The cat sat on the"))
print("Sentence 4")
print(gpt2_utils.sample_continuation("I know that my mother thinks the burglar stole"))

Sentence 1
ʻăyūʻi, and I am a Japanese scholar.

Why
Sentence 2
urchin and the second one is a piquant smell: the flavor is the same as the
Sentence 3
ground but looked up as it stared at me.
"Heh, well, it was a
Sentence 4
my home because he was tired of being in my room. So I said no to her. That


## Generating samples for labeling
Not using ngram, because the outputs are all gibberish: ngrams aren't relly well suited for autorecursive generation.

In [15]:
sentence_starts_dict = build_sentence_starts_for_sampling("../data/stimuli/pilot_object_gap.csv")

In [19]:
for sentence_id, entry in sentence_starts_dict.items():
    sentence_start = entry['sentence_start']
    
    entry['gpt2_continuations'] = [
        gpt2_utils.sample_continuation(sentence_start) 
        for _ in range(5)
    ]
    entry['grnn_continuations'] = [
        grnn_utils.sample_continuation(sentence_start) 
        for _ in range(5)
    ]

In [22]:
import spacy
nlp = spacy.load("en_core_web_sm")

for sentence_id, entry in sentence_starts_dict.items():
    sentence_start = entry['sentence_start']
    n_start_tokens = len(nlp(sentence_start))
    
    entry['gpt2_pos'] = []
    for continuation in entry['gpt2_continuations']:
        doc = nlp(sentence_start + ' ' + continuation)
        entry['gpt2_pos'].append([(token.text, token.pos_) for token in doc[n_start_tokens:]])
    
    entry['grnn_pos'] = []
    for continuation in entry['grnn_continuations']:
        doc = nlp(sentence_start + ' ' + continuation)
        entry['grnn_pos'].append([(token.text, token.pos_) for token in doc[n_start_tokens:]])

/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:36: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  hasattr(torch, "has_mps")
/home/marrsia/.local/lib/python3.8/site-packages/thinc/compat.py:37: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  and torch.has_mps  # type: ignore[attr-defined]


In [28]:
import pandas as pd

rows = []
for sentence_id, entry in sentence_starts_dict.items():
    base = {
        'sentence_id': sentence_id,
        'condition': entry['condition'],
        'embedding_level': entry['levels_of_embedding'],
        'sentence_start': entry['sentence_start'],
    }
    
    for i, (continuation, pos_tags) in enumerate(zip(entry['gpt2_continuations'], entry['gpt2_pos'])):
        rows.append({
            **base,
            'model': 'gpt2',
            'continuation_start_pos': pos_tags[0][1] if pos_tags else None,
            'continuation': continuation,
        })
    
    for i, (continuation, pos_tags) in enumerate(zip(entry['grnn_continuations'], entry['grnn_pos'])):
        rows.append({
            **base,
            'model': 'grnn',
            'continuation_start_pos': pos_tags[0][1] if pos_tags else None,
            'continuation': continuation,
        })

df = pd.DataFrame(rows)
cols = ['sentence_id', 'condition', 'embedding_level', 'model', 'continuation_start_pos', 'sentence_start', 'continuation']
df = df[cols]

df.to_csv('../data/continuations_to_label.csv', index=False)

## Analysing hand labeled continuations

In [55]:
df = pd.read_csv('../data/model_outputs/labeled_continuations.csv')

In [56]:
def unclear_summary(groupby_col):
    grouped = df.groupby([groupby_col, 'label']).size().unstack(fill_value=0)
    grouped['total'] = grouped.sum(axis=1)
    grouped['unclear_n'] = grouped.get('unclear', 0)
    grouped['unclear_pct'] = (grouped['unclear_n'] / grouped['total'] * 100).round(1)
    return grouped[['total', 'unclear_n', 'unclear_pct']]

print("=== BY MODEL ===")
print(unclear_summary('model'))

print("\n=== BY CONDITION ===")
print(unclear_summary('condition'))

print("\n=== BY EMBEDDING LEVEL ===")
print(unclear_summary('embedding_level'))

print("\n=== BY MODEL x EMBEDDING LEVEL ===")
grouped = df.groupby(['model', 'embedding_level', 'label']).size().unstack(fill_value=0)
grouped['total'] = grouped.sum(axis=1)
grouped['unclear_n'] = grouped.get('unclear', 0)
grouped['unclear_pct'] = (grouped['unclear_n'] / grouped['total'] * 100).round(1)
print(grouped[['total', 'unclear_n', 'unclear_pct']])

=== BY MODEL ===
label  total  unclear_n  unclear_pct
model                               
gpt2     120          5          4.2
grnn     120         23         19.2

=== BY CONDITION ===
label             total  unclear_n  unclear_pct
condition                                      
gap_filler          120          8          6.7
no_gap_no_filler    120         20         16.7

=== BY EMBEDDING LEVEL ===
label            total  unclear_n  unclear_pct
embedding_level                               
0                   60          5          8.3
1                   60          8         13.3
2                   60         14         23.3
3                   60          1          1.7

=== BY MODEL x EMBEDDING LEVEL ===
label                  total  unclear_n  unclear_pct
model embedding_level                               
gpt2  0                   30          1          3.3
      1                   30          1          3.3
      2                   30          3         10.0
      3   

In [57]:
clean_df = df[df['label'] != 'unclear']

counts = clean_df.groupby(['continuation_start_pos', 'label']).size().unstack(fill_value=0)
counts['total'] = counts.sum(axis=1)
pcts = counts[['gap', 'no_gap']].div(counts['total'], axis=0).mul(100).round(1)

print("=== COUNTS ===")
print(counts)
print("\n=== PERCENTAGES ===")
print(pcts)

=== COUNTS ===
label                   gap  no_gap  total
continuation_start_pos                    
ADJ                       0       2      2
ADP                      32       1     33
ADV                       3       0      3
AUX                       1       0      1
CCONJ                    12       1     13
DET                       0      26     26
NOUN                      0       4      4
NUM                       0       2      2
PRON                      0      54     54
PROPN                     0       2      2
PUNCT                    61       2     63
SCONJ                     6       0      6
SYM                       0       3      3

=== PERCENTAGES ===
label                     gap  no_gap
continuation_start_pos               
ADJ                       0.0   100.0
ADP                      97.0     3.0
ADV                     100.0     0.0
AUX                     100.0     0.0
CCONJ                    92.3     7.7
DET                       0.0   100.0
NOUN           